# MVP05 — Qualidade de Dados (diagnóstico sobre a Bronze)

Avalia as tabelas da Bronze nas dimensões de **completude, unicidade,
consistência, integridade referencial, acurácia e outliers**. Nenhum dado é
alterado aqui: cada problema encontrado gera uma regra de tratamento aplicada
no MVP06-silver.


In [0]:
from pyspark.sql import functions as F
import pandas as pd

TABELAS = ["orders", "order_items", "order_payments", "order_reviews",
           "customers", "products", "sellers", "geolocation", "category_translation"]

def bronze(t):
    return spark.table(f"mvp.bronce.{t}")

def colunas_negocio(df):
    return [c for c in df.columns if not c.startswith("_")]

In [0]:
linhas = []
for t in TABELAS:
    df = bronze(t)
    total = df.count()
    cols = colunas_negocio(df)
    nulos = df.select([
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
        for c in cols
    ]).collect()[0].asDict()
    for c, n in nulos.items():
        if n > 0:
            linhas.append({"tabela": t, "coluna": c, "nulos": n,
                           "pct_nulos": round(n / total * 100, 2)})

completude = pd.DataFrame(linhas).sort_values("pct_nulos", ascending=False)
display(completude)

In [0]:
CHAVES = {
    "orders":               ["order_id"],
    "order_items":          ["order_id", "order_item_id"],
    "order_payments":       ["order_id", "payment_sequential"],
    "order_reviews":        ["review_id"],
    "customers":            ["customer_id"],
    "products":             ["product_id"],
    "sellers":              ["seller_id"],
    "geolocation":          ["geolocation_zip_code_prefix"],
    "category_translation": ["product_category_name"],
}

linhas = []
for t, chave in CHAVES.items():
    df = bronze(t)
    total = df.count()
    distintos = df.select(*chave).distinct().count()
    linhas.append({"tabela": t, "chave": ", ".join(chave), "linhas": total,
                   "chaves_distintas": distintos, "duplicadas": total - distintos})

display(pd.DataFrame(linhas))

In [0]:
%sql
-- Pedidos com mais de uma avaliação (impacta a nota média por pedido)
SELECT COUNT(*) AS pedidos_com_multiplas_avaliacoes
FROM (SELECT order_id FROM mvp.bronce.order_reviews
      GROUP BY order_id HAVING COUNT(*) > 1);

In [0]:
TIPOS = {
    "orders":         {"order_purchase_timestamp": "TIMESTAMP", "order_approved_at": "TIMESTAMP",
                       "order_delivered_carrier_date": "TIMESTAMP",
                       "order_delivered_customer_date": "TIMESTAMP",
                       "order_estimated_delivery_date": "TIMESTAMP"},
    "order_items":    {"order_item_id": "INT", "price": "DECIMAL(10,2)",
                       "freight_value": "DECIMAL(10,2)", "shipping_limit_date": "TIMESTAMP"},
    "order_payments": {"payment_sequential": "INT", "payment_installments": "INT",
                       "payment_value": "DECIMAL(10,2)"},
    "order_reviews":  {"review_score": "INT", "review_creation_date": "TIMESTAMP"},
    "products":       {"product_weight_g": "INT", "product_length_cm": "INT",
                       "product_height_cm": "INT", "product_width_cm": "INT",
                       "product_photos_qty": "INT"},
}

linhas = []
for t, cols in TIPOS.items():
    df = spark.table(f'mvp.bronce.{t}')
    for c, tipo in cols.items():
        invalidos = df.filter(F.col(c).isNotNull() &
                              F.expr(f"try_cast({c} AS {tipo})").isNull()).count()
        linhas.append({"tabela": t, "coluna": c, "tipo_alvo": tipo,
                       "valores_invalidos": invalidos})

display(pd.DataFrame(linhas))

In [0]:
%sql
SELECT 'order_status' AS campo, order_status AS valor, COUNT(*) AS qtd
FROM mvp.bronce.orders GROUP BY order_status
UNION ALL
SELECT 'payment_type', payment_type, COUNT(*)
FROM mvp.bronce.order_payments GROUP BY payment_type
UNION ALL
SELECT 'review_score', review_score, COUNT(*)
FROM mvp.bronce.order_reviews GROUP BY review_score
ORDER BY campo, qtd DESC;

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM mvp.bronce.order_payments WHERE try_cast(payment_installments AS INT) = 0)
    AS pagamentos_com_zero_parcelas,
  (SELECT COUNT(DISTINCT customer_state) FROM mvp.bronce.customers) AS ufs_clientes,
  (SELECT COUNT(DISTINCT seller_state) FROM mvp.bronce.sellers) AS ufs_vendedores,
  (SELECT COUNT(*) FROM mvp.bronce.customers WHERE length(customer_zip_code_prefix) <> 5)
    AS ceps_cliente_fora_padrao,
  (SELECT COUNT(DISTINCT customer_city) FROM mvp.bronce.customers) AS cidades_distintas_brutas,
  (SELECT COUNT(DISTINCT lower(trim(customer_city))) FROM mvp.bronce.customers) AS cidades_apos_normalizacao_simples;

Nesta comparação mede quantas variantes de nomes de cidades são consolidadas por meio de uma normalização mínima; é a evidência numérica que justifica a padronização das cidades na camada Silver.

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM mvp.bronce.order_items i
     LEFT ANTI JOIN mvp.bronce.orders o ON i.order_id = o.order_id)          AS itens_sem_pedido,
  (SELECT COUNT(*) FROM mvp.bronce.order_items i
     LEFT ANTI JOIN mvp.bronce.products p ON i.product_id = p.product_id)    AS itens_sem_produto,
  (SELECT COUNT(*) FROM mvp.bronce.order_items i
     LEFT ANTI JOIN mvp.bronce.sellers s ON i.seller_id = s.seller_id)       AS itens_sem_vendedor,
  (SELECT COUNT(*) FROM mvp.bronce.orders o
     LEFT ANTI JOIN mvp.bronce.customers c ON o.customer_id = c.customer_id) AS pedidos_sem_cliente,
  (SELECT COUNT(*) FROM mvp.bronce.orders o
     LEFT ANTI JOIN mvp.bronce.order_items i ON o.order_id = i.order_id)     AS pedidos_sem_itens,
  (SELECT COUNT(DISTINCT p.product_category_name) FROM mvp.bronce.products p
     LEFT ANTI JOIN mvp.bronce.category_translation t
       ON p.product_category_name = t.product_category_name
     WHERE p.product_category_name IS NOT NULL)                  

Os pedidos sem itens existem e impactam a forma como a tabela fato_pedidos é construída.

In [0]:
%sql
WITH o AS (
  SELECT order_status,
         try_cast(order_purchase_timestamp      AS TIMESTAMP) AS compra,
         try_cast(order_approved_at             AS TIMESTAMP) AS aprovacao,
         try_cast(order_delivered_carrier_date  AS TIMESTAMP) AS envio,
         try_cast(order_delivered_customer_date AS TIMESTAMP) AS entrega
  FROM mvp.bronce.orders)
SELECT
  SUM(CASE WHEN aprovacao < compra THEN 1 ELSE 0 END)                               AS aprovado_antes_da_compra,
  SUM(CASE WHEN envio     < compra THEN 1 ELSE 0 END)                               AS enviado_antes_da_compra,
  SUM(CASE WHEN entrega   < compra THEN 1 ELSE 0 END)                               AS entregue_antes_da_compra,
  SUM(CASE WHEN entrega   < envio  THEN 1 ELSE 0 END)                               AS entregue_antes_do_envio,
  SUM(CASE WHEN order_status = 'delivered' AND entrega IS NULL THEN 1 ELSE 0 END)   AS delivered_sem_data,
  SUM(CASE WHEN order_status <> 'delivered' AND entrega IS NOT NULL THEN 1 ELSE 0 END) AS nao_delivered_com_data,
  MIN(compra) AS primeira_compra,
  MAX(compra) AS ultima_compra
FROM o;

In [0]:
%sql
SELECT date_format(try_cast(order_purchase_timestamp AS TIMESTAMP), 'yyyy-MM') AS ano_mes,
       COUNT(*) AS pedidos
FROM mvp.bronce.orders
GROUP BY 1 ORDER BY 1;.0``

Ao inicio e ao final do DW tem poucos pedidos.

In [0]:
def perfil_outliers(tabela, coluna, tipo="DOUBLE"):
    v = bronze(tabela).select(F.expr(f"try_cast({coluna} AS {tipo})").alias("x")).dropna()
    q1, med, q3 = v.approxQuantile("x", [0.25, 0.5, 0.75], 0.001)
    lim = q3 + 1.5 * (q3 - q1)
    agg = v.agg(F.min("x").alias("minimo"), F.max("x").alias("maximo"),
                F.count("x").alias("n"),
                F.sum(F.when(F.col("x") > lim, 1).otherwise(0)).alias("acima")).collect()[0]
    return {"tabela": tabela, "coluna": coluna, "minimo": agg.minimo,
            "q1": q1, "mediana": med, "q3": q3,
            "limite_superior_iqr": round(lim, 2), "maximo": agg.maximo,
            "qtd_outliers": agg.acima, "pct_outliers": round(agg.acima / agg.n * 100, 2)}

outliers = pd.DataFrame([
    perfil_outliers("order_items", "price"),
    perfil_outliers("order_items", "freight_value"),
    perfil_outliers("order_payments", "payment_value"),
    perfil_outliers("order_payments", "payment_installments"),
    perfil_outliers("products", "product_weight_g"),
])
display(outliers)

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM mvp.bronce.order_items WHERE try_cast(price AS DOUBLE) <= 0)            AS precos_nao_positivos,
  (SELECT COUNT(*) FROM mvp.bronce.order_items WHERE try_cast(freight_value AS DOUBLE) = 0)     AS itens_frete_zero,
  (SELECT COUNT(*) FROM mvp.bronce.products    WHERE try_cast(product_weight_g AS INT) = 0)     AS produtos_peso_zero;

## Síntese: achados → tratamento na Silver

| # | Tabela | Dimensão | Achado | Valor | Tratamento no MVP06 |
|---|---|---|---|---|---|
| 1 | orders | Completude | `order_delivered_customer_date` nulo | 2.98 % | Mantido nulo: pedido não entregue. Excluído apenas das métricas de prazo. |
| 2 | orders | Acurácia | Status `delivered` sem data de entrega | 8 linhas | Mantido, sinalizado; fora das métricas de prazo. |
| 3 | orders | Acurácia | Meses extremos com poucos pedidos | 4 meses | Mantidos na Silver; excluídos da série temporal da pergunta 5. |
| 4 | order_reviews | Unicidade | Pedidos com mais de uma avaliação | 547 pedidos | Mantida a avaliação mais recente por pedido. |
| 5 | order_reviews | Completude | Título e comentário vazios | 88.34 % e 58.71% | Esperado (campos opcionais). Sem tratamento. |
| 6 | products | Completude | Categoria nula | 610 produtos | Substituída por "sem_categoria". |
| 7 | products | Consistência | Categorias sem tradução | 2 categorias | `categoria_en` recebe o nome em português. |
| 8 | products | Acurácia | Peso ou dimensões nulos/zero | 8 produtos nulos e 4 produtos peso zero| Convertidos para nulo; volume calculado só quando completo. |
| 9 | customers/sellers | Consistência | Variações de grafia de cidade | 4119 → 4119 cidades | Remoção de acentos, trim e capitalização. |
| 10 | order_payments | Consistência | `payment_type = not_defined` | 3 linhas | Mantido; não é escolhido como meio principal. |
| 11 | order_payments | Consistência | Parcelas = 0 | 2 linhas | Convertido para 1 (pagamento à vista). |
| 12 | orders × order_items | Integridade | Pedidos sem itens | 775 pedidos | Mantidos na fato com valores nulos; excluídos das análises de receita. |
| 13 | geolocation | Unicidade | Múltiplas coordenadas por CEP | 981148 | Esperado pela natureza da tabela. Fora do modelo Gold (olhar MVP03). |
| 14 | order_items | Outliers | Preços acima do limite IQR | 7.48% | Mantidos: valores legítimos. Análises usam mediana quando pertinente. |